## Module_3: *Lung Fibrosis*

## Team Members:
*Tahseen Azad*, 
*Aryan Mhaskar*

## Project Title:
*Determining Different Biopsy Depths of Lung Fibrosis*



## Project Goal:
This project seeks develop an image analysis pipeline to determine the extent of lung fibrosis in patients. In order to do this, we will interpolate the amount of fibrosis at a speicifc depth by creating an algorithm using a given data set.

## Disease Background: 
*Fill in information and please note that this module is truncated and only has 5 bullets (instead of the 11 that you did for Module #1).*

* Prevalence & incidence: Lung fibrosis is a relatively rare but serious chronic disease, affecting ~3–9 per 100,000 people annually in the U.S., with higher prevalence in older adults (typically >50 years).
* Risk factors (genetic, lifestyle): Key risk factors include aging, smoking, environmental exposures (e.g., dust, asbestos), and genetic predispositions (mutations affecting telomere maintenance or surfactant proteins).
* Symptoms: Common symptoms include progressive shortness of breath, persistent dry cough, fatigue, and reduced exercise tolerance; advanced stages may show clubbing of fingers and hypoxemia.
* Standard of care treatment(s): Treatments include antifibrotic drugs (e.g., pirfenidone, nintedanib) to slow disease progression, supplemental oxygen therapy, pulmonary rehabilitation, and lung transplantation in severe cases.
* Biological mechanisms (anatomy, organ physiology, cell & molecular physiology): Lung fibrosis involves excessive deposition of extracellular matrix (especially collagen) in the interstitial lung tissue, driven by abnormal wound healing, fibroblast activation, and epithelial cell injury which leads to stiffened lungs and impaired gas exchange.

## Data-Set: 
Our data set is given by: 
"Unpublished data was collected by the Peirce-Cottler Lab (Dept. of Biomedical Engineering) and Kim Lab (Division of Pulmonary and Critical Care) at the University of Virginia School of Medicine. 

Also, since we covered this in Lecture 1 of this Module in detail, describe how the data was collected -- What techniques were used? What units are the data measured in? Etc.*

The data used in this module was collected from fibrotic mouse lung. The fibrotic mouse lung was prepared using a bleomycin-induced lung injury model. Bleomycin is an antiobiotic isolated from *Streptomyces verticillus* usually used in chemotherapy to treat various types of cancer. However, bleomycin is also known to induce ideopathic pulmonary fibrosis. Bleomycin was injected into the trachea of our mouse model and the lung was harvested after three weeks. After, the lungs are fixed with paraformaldehyde and mounted in wax. Then, using a cryotome, tissue slices are produced and placed in glass slides. From there, the tissues were immunostained with desmin, smooth muscle alpha actin (stains for large blood vessels in smooth muscle), and CD-31 (endothelial cells in all blood vessels including capillaries). Desmin signal indicates prescence of myofibroblasts which are cells that make fibrotic scar tissue that forms leasions. These stains were visualized in black and white images where white represents lesion tissue at diffeent depths of the lungs. 



## Data Analyis: 
*Describe how you analyzed the data. This is where you should intersperse your Python code so that anyone reading this Jupyter notebook can run your code to perform the analysis that you did, generate your figures, generate your .csv file, etc.). Show your graphs here, which should have proper labeles (e.g., x- and y-axes labels). Each graph you present should be thoroughly described by a caption so reader understands the data, why you are presenting it, and the main conclusion fromt the data.*

In [ ]:
"""
Module 3: Count black and white pixels, compute the percentage of white pixels
in .jpg images, and write results to a .csv file.
"""

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import time

start_time = time.perf_counter()

# ── Configuration ─────────────────────────────────────────────────────────────

FILENAMES = [
    r"../images/MASK_SK658 Llobe ch010039.jpg",
    r"../images/MASK_SK658 Slobe ch010066.jpg",
    r"../images/MASK_SK658 Slobe ch010147.jpg",
    r"../images/MASK_SK658 Slobe ch010110.jpg",
    r"../images/MASK_SK658 Slobe ch010130.jpg",
    r"../images/MASK_SK658 Slobe ch010114.jpg",
]

# Depths (microns) corresponding to each image above
DEPTHS = [15, 1000, 3000, 5300, 7000, 9900]

OUTPUT_CSV = "Percent_White_Pixels.csv"

# ── Image Processing ───────────────────────────────────────────────────────────

def analyze_image(filename):
    """
    Load a grayscale image, threshold it to binary, and return
    (white_count, black_count, white_percent).
    Raises FileNotFoundError if the image cannot be loaded.
    """
    img = cv2.imread(filename, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Could not load image: {filename}")

    _, binary = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)

    white = int(np.sum(binary == 255))
    black = int(np.sum(binary == 0))
    total = white + black
    white_pct = 100.0 * white / total if total > 0 else 0.0

    return white, black, white_pct


if len(FILENAMES) != len(DEPTHS):
    raise ValueError("FILENAMES and DEPTHS must be the same length.")

results = []

# ── Analyze each image ─────────────────────────────────────────────────────────
print("=" * 60)
print("Pixel counts per image")
print("=" * 60)

for filename, depth in zip(FILENAMES, DEPTHS):
    white, black, white_pct = analyze_image(filename)
    results.append({
        "Filename": filename,
        "Depth (microns)": depth,
        "White pixels": white,
        "Black pixels": black,
        "White percent": white_pct,
    })
    print(f"\n  File : {filename}")
    print(f"  Depth: {depth} microns")
    print(f"  White: {white:,} px  |  Black: {black:,} px  |  {white_pct:.2f}% white")

# ── Write CSV ──────────────────────────────────────────────────────────────────
df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)

print("\n" + "=" * 60)
print(f"Results written to '{OUTPUT_CSV}'")
print("=" * 60)

end_time = time.perf_counter()
elapsed_time = end_time - start_time

print(f"Code block took: {elapsed_time:.4f} seconds")

# ── Extract depths and white percentages from results ─────────────────────────
depths = [r["Depth (microns)"] for r in results]
white_percents = [r["White percent"] for r in results]

# ── Interpolate a point ────────────────────────────────────────────────────────
interpolate_depth = float(input(
    "Enter the depth at which you want to interpolate a point (in microns): "))

x = depths
y = white_percents

i = interp1d(x, y, kind='linear')
interpolate_point = float(i(interpolate_depth))
print(f"The interpolated point is at the x-coordinate {interpolate_depth} "
      f"and y-coordinate {interpolate_point}.")

depths_i = depths[:]
depths_i.append(interpolate_depth)
white_percents_i = white_percents[:]
white_percents_i.append(interpolate_point)

# ── Plotting ───────────────────────────────────────────────────────────────────
fig, axs = plt.subplots(2, 1)

axs[0].scatter(depths, white_percents, marker='o', color='blue')
axs[0].set_title('Plot of depth of image vs percentage white pixels')
axs[0].set_xlabel('depth of image (in microns)')
axs[0].set_ylabel('white pixels as a percentage of total pixels')
axs[0].grid(True)

axs[1].scatter(depths_i, white_percents_i, marker='o', color='blue')
axs[1].set_title(
    'Plot of depth of image vs percentage white pixels with interpolated point (in red)')
axs[1].set_xlabel('depth of image (in microns)')
axs[1].set_ylabel('white pixels as a percentage of total pixels')
axs[1].grid(True)
axs[1].scatter(depths_i[-1], white_percents_i[-1],
               color='red', s=100, label='Highlighted point')
axs[1].legend()

plt.tight_layout()
plt.show()

## Verify and validate your analysis: 
*Describe how you checked to see that your analysis gave you an answer that you believe (verify). Describe how your determined if your analysis gave you an answer that is supported by other evidence (e.g., by comparing your analysis to a published paper).*

## Conclusions and Ethical Implications: 
*Think about the answer your analysis generated, draw conclusions related to your overarching question, and discuss the ethical implications of your conclusions.*

## Limitations and Future Work: 
*Describe the limitations of your project. If you had more time to work on this, what would you do to explore further or refine your analysis?*

## References:
*You can use any format you like but provide the citations for facts that you referenced in this project notebook.*